In [ ]:
import os
from dotenv import load_dotenv


In [ ]:
load_dotenv()

True

In [ ]:
if os.environ['OPENAI_API_KEY']:
    print("OPENAI key is available")

OPENAI key is available


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

In [ ]:
llm = ChatOpenAI(model = "gpt-5-nano", temperature=0)

In [ ]:
response = llm.invoke("What is capital of India")

In [ ]:
response.content

'New Delhi. It is the capital of India, located in the National Capital Territory of Delhi.'

# RAG implementation with PDF

### Step 1: Extracting text from PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./Docs/Harry_Potter.pdf"

loader = PyPDFLoader(pdf_path)

docs = loader.load()

In [ ]:
docs

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-04-22T12:03:44+05:30', 'author': 'himanshu bansal', 'moddate': '2026-04-22T12:03:44+05:30', 'source': './Docs/Harry_Potter.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Harry Potter Film Series — Overview \nThe Harry Potter film series is an eight-part fantasy saga following the life of a young wizard, Harry \nPotter, as he discovers his magical identity and attends Hogwarts School of Witchcraft and \nWizardry. Alongside his closest friends, Ron Weasley and Hermione Granger, Harry navigates \nfriendship, danger, and destiny. As the story progresses, the tone shifts from magical adventure to \na darker battle between good and evil, centered around the rise of Lord Voldemort. The films \nexplore themes of courage, loyalty, sacrifice, and the enduring power of love. The series is one of \nthe most successful film franchises globally, known for its immersi

In [ ]:
# Optional - create your onw metadata

for i in docs:
    i.metadata = {"source": "pdf",
                  "developer": "shubham"}

### Step 2: chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'pdf', 'developer': 'shubham'}, page_content='Harry Potter Film Series — Overview \nThe Harry Potter film series is an eight-part fantasy saga following the life of a young wizard, Harry'),
 Document(metadata={'source': 'pdf', 'developer': 'shubham'}, page_content='Potter, as he discovers his magical identity and attends Hogwarts School of Witchcraft and \nWizardry. Alongside his closest friends, Ron Weasley and Hermione Granger, Harry navigates'),
 Document(metadata={'source': 'pdf', 'developer': 'shubham'}, page_content='friendship, danger, and destiny. As the story progresses, the tone shifts from magical adventure to \na darker battle between good and evil, centered around the rise of Lord Voldemort. The films'),
 Document(metadata={'source': 'pdf', 'developer': 'shubham'}, page_content='explore themes of courage, loyalty, sacrifice, and the enduring power of love. The series is one of \nthe most successful film franchises globally, known for its immer

In [ ]:
len(chunks)

9

### Step 3: Creating embeddings and storing in vector database

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small")

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### Step 4: Semantic search

In [ ]:
vectorstore.similarity_search("Which Harry Potter movie earned $1B+ from Box office?", k=3)

[Document(metadata={'developer': 'shubham', 'source': 'pdf'}, page_content='4. Goblet of Fire (2005) — IMDb: 7.7 | Duration: 157 min | Box Office: $0.90B \n5. Order of the Phoenix (2007) — IMDb: 7.5 | Duration: 138 min | Box Office: $0.94B'),
 Document(metadata={'source': 'pdf', 'developer': 'shubham'}, page_content='6. Half-Blood Prince (2009) — IMDb: 7.6 | Duration: 153 min | Box Office: $0.93B \n7. Deathly Hallows – Part 1 (2010) — IMDb: 7.7 | Duration: 146 min | Box Office: $0.98B'),
 Document(metadata={'developer': 'shubham', 'source': 'pdf'}, page_content='2. Chamber of Secrets (2002) — IMDb: 7.4 | Duration: 161 min | Box Office: $0.88B \n3. Prisoner of Azkaban (2004) — IMDb: 7.9 | Duration: 142 min | Box Office: $0.80B')]

In [ ]:
context = vectorstore.similarity_search("Which Harry Potter movie earned $1B+ from Box office?", k=3)

In [ ]:
response = llm.invoke(f"Which Harry Potter movie earned $1B+ from Box office? use only this context and no llm search outside this: {context}")

In [ ]:
response.content


'None. Based on the provided data, no Harry Potter movie earned $1B+ at the box office. The highest listed is Deathly Hallows – Part 1 (2010) with about $0.98B.'

## Git test 2

## Git test 4

## Git test 3